# Using IBM watsonx.governance - monitoring the models deployed in Azure Databricks workspace

In [0]:
%pip install ibm_watson_machine_learning
!pip install ibm_cloud_sdk_core
!pip install ibm_watson_openscale
!pip install python-dotenv
!pip install ibm-aigov-facts-client

In [0]:
%restart_python

In [0]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

from ibm_aigov_facts_client import AIGovFactsClient, CloudPakforDataConfig, DetachedPromptTemplate, PromptTemplate, DeploymentDetails, ModelDetails

### Cloud Pak for Data Credentials

In [0]:
load_dotenv(".env")
CPD_URL = os.getenv("CPD_URL")
CPD_USERNAME = os.getenv("CPD_USERNAME")
CPD_PASSWORD = os.getenv("CPD_PASSWORD")
CPD_APIKEY= os.getenv("CPD_APIKEY")

#masked
WOS_CREDENTIALS = {
    "url": CPD_URL,
    "username": CPD_USERNAME,
    "password": CPD_PASSWORD,
    "apikey": CPD_APIKEY,
    "version": "5.0"
}

In [0]:
WML_CREDENTIALS = WOS_CREDENTIALS.copy()
WML_CREDENTIALS['instance_id']='openshift'
WML_CREDENTIALS

### Construct the scoring payload 

In [0]:
val_df= pd.read_csv("val.csv")

In [0]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

def get_scoring_payload(no_of_records_to_score = 1):

    fields = ["lag_2", "lag_1"]
    values = val_df[fields].values.tolist()
    payload_scoring = {"fields": fields, "values": values[:no_of_records_to_score]}
    return payload_scoring

In [0]:
model_payload= get_scoring_payload(no_of_records_to_score=5)
model_payload

# IBM watsonx.governance - monitoring Configuration

## Get the training data

In [0]:
import pandas as pd
training_data = pd.read_csv("train.csv", sep=",", header=0)
training_data.head()

In [0]:
len(training_data)

## Describe the training data - features, categorical columns, label column

In [0]:
scoring_payload= model_payload
input_fields = scoring_payload['fields']
feature_columns = input_fields
cat_features = []
class_label= "y"
prediction_field='prediction'


## IBM watsonx.governance Authentication

In [0]:
from ibm_cloud_sdk_core.authenticators import CloudPakForDataAuthenticator
from ibm_watson_openscale import APIClient

from ibm_watson_openscale import *
from ibm_watson_openscale.supporting_classes.enums import *
from ibm_watson_openscale.supporting_classes import *
from ibm_watson_openscale.utils import *
from ibm_watson_openscale.base_classes.watson_open_scale_v2 import *

In [0]:
authenticator = CloudPakForDataAuthenticator(
        url=WML_CREDENTIALS['url'],
        username=WML_CREDENTIALS['username'],
        apikey=WML_CREDENTIALS['apikey'],
        disable_ssl_verification=True
    )
wos_client = APIClient(service_url=WML_CREDENTIALS['url'],authenticator=authenticator)
wos_client.version

In [0]:
wos_client.data_marts.show()

## Use the existing data mart

In [0]:
data_marts = wos_client.data_marts.list().result.data_marts
data_mart_id=data_marts[0].metadata.id
print('Using existing datamart {}'.format(data_mart_id))

## Create a service provider to WML Python Function that wraps the ADB model deployment

In [0]:
SERVICE_PROVIDER_NAME = "Azure Databricks Provider Prod 2"
SERVICE_PROVIDER_DESCRIPTION = "Monitoring the model deployed in Azure Databricks using a custom ML Provider deployed on Azure"

scoring_url= "https://stock-prediction-custom-ml-provider.azurewebsites.net/predictions"
asset_deployment_scoring_url = scoring_url

scoring_endpoint_url = scoring_url
scoring_request_headers = {
        "Content-Type": "application/json",
    }

### Cleanup of the service providers

In [0]:
service_providers = wos_client.service_providers.list().result.service_providers
for service_provider in service_providers:
    service_instance_name = service_provider.entity.name
    if service_instance_name == SERVICE_PROVIDER_NAME:
        service_provider_id = service_provider.metadata.id
        wos_client.service_providers.delete(service_provider_id)
        print("Deleted existing service_provider for WML instance: {}".format(service_provider_id))

## Create the service provider here

In [0]:
# request_headers = {"Content-Type": "application/json"}
CUSTOM_ENGINE_CREDENTIALS = {
    "url": scoring_url,
    "username": "user",
    "password": "password",
}

added_service_provider_result = wos_client.service_providers.add(
        name=SERVICE_PROVIDER_NAME,
        description=SERVICE_PROVIDER_DESCRIPTION,
        service_type=ServiceTypes.CUSTOM_MACHINE_LEARNING,
        # request_headers=request_headers,
        # operational_space_id = "pre_production",
        operational_space_id = "production",
        credentials=CustomCredentials(
            url= CUSTOM_ENGINE_CREDENTIALS['url'],
            username= CUSTOM_ENGINE_CREDENTIALS['username'],
            password= CUSTOM_ENGINE_CREDENTIALS['password'],
        ),
        background_mode=False
    ).result
service_provider_id = added_service_provider_result.metadata.id

In [0]:
wos_client.service_providers.show()

In [0]:
service_provider_id= "019935ed-e893-7ff1-b283-b4ac2653ae9d" #	Azure Databricks Provider Prod 2

## Subsribe the Python Function Deployment with IBM watsonx.governance - monitoring

In [0]:
wos_client.subscriptions.show()

### Cleanup of the subscription

In [0]:
import uuid
asset_id = str(uuid.uuid4())
url = ''

experiment_name= "amazon-stock-price-0919"
SUBSCRIPTION_NAME = f"[Asset] {experiment_name}"
asset_name = SUBSCRIPTION_NAME
asset_deployment_id = "id"
asset_deployment_name = f"{experiment_name}-deployment"


In [0]:
subscriptions = wos_client.subscriptions.list().result.subscriptions
for subscription in subscriptions:
    if subscription.entity.asset.name == SUBSCRIPTION_NAME:
        sub_model_id = subscription.metadata.id
        wos_client.subscriptions.delete(subscription.metadata.id)
        print('Deleted existing subscription for model', sub_model_id)

### Create the subscription here

In [0]:
subscription_details = wos_client.subscriptions.add(
        data_mart_id=data_mart_id,
        service_provider_id=service_provider_id,
        asset=Asset(
            asset_id=asset_id,
            name=asset_name,
            url=url,
            asset_type=AssetTypes.MODEL,
            input_data_type=InputDataType.STRUCTURED,
            problem_type=ProblemType.REGRESSION
        ),
        deployment=AssetDeploymentRequest(
            deployment_id=asset_deployment_id,
            name=asset_deployment_name,
            deployment_type= DeploymentTypes.ONLINE,
            scoring_endpoint=ScoringEndpointRequest(
                url=scoring_endpoint_url,
                request_headers=scoring_request_headers
            )
        ),
        asset_properties=AssetPropertiesRequest(
            label_column=class_label,
            prediction_field=prediction_field,
            feature_fields = feature_columns,
            categorical_fields = cat_features,
        #     training_data_reference=TrainingDataReference(type='cos',
        #                                                   location=COSTrainingDataReferenceLocation(bucket = BUCKET_NAME,
        #                                                                                             file_name = training_data_file_name),
        #                                                   connection=COSTrainingDataReferenceConnection.from_dict({
        #                                                       "resource_instance_id": COS_RESOURCE_CRN,
        #                                                       "url": COS_ENDPOINT,
        #                                                       "api_key": COS_API_KEY_ID,
        #                                                       "iam_url": IAM_URL})),            
        ),
        background_mode= False
    ).result

In [0]:
subscription_id = subscription_details.metadata.id
subscription_id

## Get the payload logging data set id, to where the payload logging is done.

In [0]:
import time

time.sleep(5)
payload_data_set_id = None
payload_data_set_id = wos_client.data_sets.list(type=DataSetTypes.PAYLOAD_LOGGING, 
                                                target_target_id=subscription_id, 
                                                target_target_type=TargetTypes.SUBSCRIPTION).result.data_sets[0].metadata.id
if payload_data_set_id is None:
    print("Payload data set not found. Please check subscription status.")
else:
    print("Payload data set id: ", payload_data_set_id)

In [0]:
scoring_payload = get_scoring_payload(no_of_records_to_score=100)
scoring_payload

Score Azure model deployment

In [0]:

from requests.auth import HTTPBasicAuth
scoring_response= requests.post(scoring_url, json= scoring_payload, auth=HTTPBasicAuth('user', 'password')).json()
scoring_response

## Payload logging 

In [0]:
from ibm_watson_openscale.supporting_classes.payload_record import PayloadRecord
def payload_logging(payload_scoring, scoring_response):
    scoring_id = str(uuid.uuid4())
    records_list=[]
    
    #manual PL logging for custom ml provider
    pl_record = PayloadRecord(scoring_id=scoring_id, request=payload_scoring, response=scoring_response, response_time=int(460))
    records_list.append(pl_record)
    wos_client.data_sets.store_records(data_set_id = payload_data_set_id, request_body=records_list)
    
    time.sleep(10)
    pl_records_count = wos_client.data_sets.get_records_count(payload_data_set_id)
    print("Number of records in the payload logging table: {}".format(pl_records_count))
    return scoring_id

In [0]:
scoring_id = payload_logging(scoring_payload, scoring_response)
print('scoring_id: ' + str(scoring_id))

### Make sure the records reached the payload logging table

In [0]:
import uuid
from ibm_watson_openscale.supporting_classes.payload_record import PayloadRecord
pl_records_count = wos_client.data_sets.get_records_count(payload_data_set_id)
print("Number of records in the payload logging table: {}".format(pl_records_count))
if pl_records_count == 0:
    raise Exception("Payload logging did not happen!")

In [0]:
wos_client.data_sets.show_records(payload_data_set_id, limit= 5)

# Configure Quality Monitoring

In [0]:
import time

target = Target(
        target_type=TargetTypes.SUBSCRIPTION,
        target_id=subscription_id
)
parameters = {
    "min_feedback_data_size": 10
}
quality_monitor_details = wos_client.monitor_instances.create(
    data_mart_id=data_mart_id,
    background_mode=False,
    monitor_definition_id=wos_client.monitor_definitions.MONITORS.QUALITY.ID,
    target=target,
    parameters=parameters
).result

## Get the quality monitor instance id and the feedback data set id

In [0]:
quality_monitor_instance_id = quality_monitor_details.metadata.id
quality_monitor_instance_id

In [0]:
feedback_dataset_id = None
feedback_dataset = wos_client.data_sets.list(type=DataSetTypes.FEEDBACK, 
                                                target_target_id=subscription_id, 
                                                target_target_type=TargetTypes.SUBSCRIPTION).result
print(feedback_dataset)
feedback_dataset_id = feedback_dataset.data_sets[0].metadata.id
if feedback_dataset_id is None:
    print("Feedback data set not found. Please check quality monitor status.")

In [0]:
feedback_dataset_id

## Perform feedback logging

In [0]:
feedback_data = pd.read_csv("feedback_100.csv", sep=",", header=0)
feedback_data.head()

In [0]:
cols_to_remove = []
def get_feedback_payload():
    for col in cols_to_remove:
        if col in feedback_data.columns:
            del feedback_data[col]

    fields = feedback_data.columns.tolist()
    values = feedback_data[fields].values.tolist()

    feedback_payload = {"fields": fields, "values": values}
    return feedback_payload

In [0]:
feedback_payload = get_feedback_payload()
feedback_payload

### Load the feedback data

In [0]:
wos_client.data_sets.store_records(feedback_dataset_id, request_body=[feedback_payload], background_mode=False)

In [0]:
wos_client.data_sets.get_records_count(data_set_id=feedback_dataset_id)

## Create the MRM monitor

In [0]:
# wos_client.monitor_definitions.show()
wos_client.monitor_instances.show(target_target_id= subscription_id)

In [0]:
target = Target(
        target_type=TargetTypes.SUBSCRIPTION,
        target_id=subscription_id
)
parameters = {
    "min_feedback_data_size": 10
}
mrm_monitor_details = wos_client.monitor_instances.create(
    data_mart_id=data_mart_id,
    background_mode=False,
    monitor_definition_id="mrm",
    target=target,
    parameters=parameters
).result

In [0]:
mrm_monitor_details = mrm_monitor_details.metadata.id
mrm_monitor_details

## Trigger the MRM monitor (evaluates all monitors)

In [0]:
mrm_monitor_instance_id = "01996374-79b0-7d17-a576-ac0efb8c8831"

response = wos_client.monitor_instances.mrm.evaluate_risk(monitor_instance_id=mrm_monitor_instance_id, 
                                                    body = {},
                                                    # evaluation_tests = ["quality", "model_health"],
                                                    background_mode = False)

## Trigger the quality monitor

In [0]:
run_details = wos_client.monitor_instances.run(monitor_instance_id=quality_monitor_instance_id, background_mode=False).result

### Fetch the quality monitor evaluated metrics

In [0]:
wos_client.monitor_instances.show_metrics(monitor_instance_id=quality_monitor_instance_id)

### Go to External models on watsonx.governance and track the new model asset to a use case